# 基于pytorch的高层API实现简单线性回归模型
* 任务描述：</br>
将“机器学习概述-上.ipynb” 中的生成样本数据的`create_toy_data`函数改为用pytorch实现，利用该`create_toy_data`函数，按要求生成带噪音的样本，并采用pytorch的高层API来实现线性回归。

In [ ]:
import datetime
import os

def printLog(name, ID):
    # 打印和
    print(': %s; : %s' % (name, ID))
    # 打印操作系统的用户名
    print("系统用户名:", os.getlogin())
    # 打印当前时间
    print('运行时间:', datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

"""*********************************************************************
   【注意】运行前，请把下面代码中的“张三”改为自己的，123456改为自己的。
   *********************************************************************"""
printLog('', )

# 0. 相关类说明
## 0.1 torch.nn.Linear（用于模型定义）
__`torch.nn.Linear(in_features, out_features, bias=True, device=None, dtype=None)`__  </br>
`torch.nn.Linear` 用于实现全连接层（也称为线性层或密集层）。全连接层是神经网络中最基本的层之一，它的功能是将输入数据进行线性变换，通常用于网络的最后几层，以输出最终的预测结果。</br>
主要参数：
* in_features (int)：输入样本的特征数量。
* out_features (int)：输出样本的特征数量，也就是线性变换后的维度。
* bias (bool, optional)：如果设置为False，则该层不会学习附加的偏置项。默认情况下是True。
* device (torch.device, optional)：指定模型参数所在的设备（CPU或GPU）。
* dtype (torch.dtype, optional)：指定模型参数的数据类型。

下面是一个使用`torch.nn.Linear`构建简单线性回归模型的例子：

In [ ]:
import torch
import torch.nn as nn

# 定义一个简单的线性回归模型
class SimpleLinearRegression(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(SimpleLinearRegression, self).__init__()
        self.linear = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        return self.linear(x)

# 实例化模型
input_dim = 1  # 输入特征的数量
output_dim = 1  # 输出特征的数量
model = SimpleLinearRegression(input_dim, output_dim)

# 创建一个输入张量
x_input = torch.tensor([[1.0], [2.0], [3.0]], dtype=torch.float32)

# 前向传播
output = model(x_input)
print("Output:", output)

# 检查模型的参数
print("Model's parameters:")
for name, param in model.named_parameters():
    print(f"{name}: {param}")

## 0.2 torch.nn.MSELoss（损失函数）
__`torch.nn.MSELoss(size_average=None, reduce=None, reduction='mean')`__</br>
`torch.nn.MSELoss`是PyTorch中用于计算均方误差（Mean Squared Error, MSE）损失的一个模块。MSE损失常常用于回归任务中，衡量预测值与真实值之间的差异。</br>
主要参数：
* size_average (bool, optional)：如果指定，则损失将在批量中平均；如果None并且reduce也是None，则会根据reduction参数进行处理。此参数已被弃用，不应再使用。
* reduce (bool, optional)：如果指定，则对于批量中的所有损失，将它们全部求和或取平均；如果None，则根据reduction参数处理。此参数已被弃用，不应再使用。
* reduction (string, optional)：指定损失的聚合方式。选项包括：</br>
'none'：没有聚合，返回每一对输入的损失值。</br>
'mean'：损失值的均值。</br>
'sum'：损失值的总和。 默认为'mean'。</br>

下面是一个简单的例子来展示如何使用 torch.nn.MSELoss：

In [ ]:
import torch
import torch.nn as nn

# 创建一个MSELoss实例
loss_fn = nn.MSELoss()

# 假设我们有一些预测值和相应的标签
predictions = torch.tensor([[0.1], [0.6], [0.7]], dtype=torch.float32)
labels = torch.tensor([[0.0], [0.5], [0.9]], dtype=torch.float32)

# 计算MSE损失
loss = loss_fn(predictions, labels)

# 打印损失值
print("MSE Loss:", loss.item())

## 0.3 torch.optim.Adam（优化器）
__`torch.optim.Adam(params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0, amsgrad=False)`__</br>
`torch.optim.Adam` 是 PyTorch 中实现 Adam 优化算法的一个优化器类。Adam（Adaptive Moment Estimation）是一种自适应学习率的方法，它结合了动量（Momentum）和 RMSProp 的概念，广泛应用于深度学习中的模型参数优化,能够有效地调整每个参数的学习率，从而加速收敛过程。</br>
主要参数：
* params (iterable): 待优化参数的迭代器，通常是模型参数的迭代器。
* lr (float, 可选): 初始学习率，默认为 1e-3。
* betas (tuple, 可选): 用于计算运行平均值和运行方差的系数，默认为 (0.9, 0.999)。</br>
  betas[0] 对应 beta1，即动量项的衰减因子。</br>
  betas[1] 对应 beta2，即二阶矩估计的衰减因子。</br>
* eps (float, 可选): 添加到分母中的项以提高数值稳定性，用于防止分母为零的数值，默认为 1e-8。
* weight_decay (float, 可选): 权重衰减（L2 正则化）系数，默认为 0。
* amsgrad (bool, 可选): 是否使用 AMSGrad 的修正，默认为 False。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# 定义一个简单的线性模型
class LinearModel(nn.Module):
    def __init__(self):
        super(LinearModel, self).__init__()
        self.linear = nn.Linear(in_features=10, out_features=1)

    def forward(self, x):
        return self.linear(x)

# 实例化模型
model = LinearModel()

# 假设的输入和标签
inputs = torch.randn(32, 10, dtype=torch.float32)
labels = torch.randn(32, 1, dtype=torch.float32)

# 定义损失函数
criterion = nn.MSELoss()
# 定义 Adam 优化器，指定要优化的参数为模型的所有参数
optimizer = optim.Adam(model.parameters())
# 前向传播
preds = model(inputs)
# 计算损失
loss = criterion(preds, labels)
# 反向传播
loss.backward()
# 更新参数
optimizer.step()
# 清除梯度，为下一次迭代做准备
optimizer.zero_grad()

print("Loss:", loss.item())

# 1. 数据集构建

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号

def linear_func(x):
    return 1.5 * x + 0.6

def create_toy_data(func, sample_size, std=0.2, random_seed=42):
    np.random.seed(random_seed)
    x = np.random.uniform(0, 1, sample_size)
    y = func(x) + np.random.normal(scale=std, size=sample_size)
    return x, y

X_train, y_train = create_toy_data(linear_func, 500)
X_test, y_test = create_toy_data(linear_func, 200)

# 绘制数据
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.scatter(X_train, y_train, alpha=0.5, label='训练数据')
plt.plot(np.linspace(0, 1, 100), linear_func(np.linspace(0, 1, 100)), 'r-', label='真实函数')
plt.xlabel('x')
plt.ylabel('y')
plt.title('训练数据分布')
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(X_test, y_test, alpha=0.5, label='测试数据', color='orange')
plt.plot(np.linspace(0, 1, 100), linear_func(np.linspace(0, 1, 100)), 'r-', label='真实函数')
plt.xlabel('x')
plt.ylabel('y')
plt.title('测试数据分布')
plt.legend()
plt.show()

In [ ]:
"执行下面的基于Paddle API实现的模型定义和训练前，要按照下面的方法重新修改训练数据X_train、y_train的shape，为什么？"
# 转换为 PyTorch Tensor
x = X_train.reshape([-1, 1])
y = y_train.reshape([-1, 1])
print('x:', x)
print('y:', y)

# 2. 基于Pytorch高层API的线性回归模型定义

In [ ]:
# 使用torch.nn.Linear定义线性回归模型LinearRegression()
class LinearRegression(nn.Module):
    def __init__(self):
        super(LinearRegression, self).__init__()
        self.linear = nn.Linear(1, 1)  # 输入特征维度为1，输出维度为1
    
    def forward(self, x):
        return self.linear(x)

# 3. 基于Pytorch高层API的模型训练

In [ ]:
# 基于Pytorch高层API的模型训练
# 将numpy数组转换为PyTorch Tensor
x_tensor = torch.tensor(x, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

# 实例化模型、损失函数和优化器
model = LinearRegression()
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.1)

# 训练模型
epochs = 200
for epoch in range(epochs):
    # 前向传播
    pred = model(x_tensor)
    loss = loss_fn(pred, y_tensor)
    
    # 反向传播和优化
    optimizer.zero_grad()  # 清除梯度
    loss.backward()        # 反向传播计算梯度
    optimizer.step()       # 更新参数
    
    # 打印损失
    if (epoch+1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.6f}")

# 单个数据的预测
x_new = np.array([[5.0]], dtype='float32')
x_new_tensor = torch.tensor(x_new, dtype=torch.float32)
model.eval()
with torch.no_grad():
    pred_new = model(x_new_tensor)
print(f"Predicted value: {pred_new.numpy()}")  # 输出模型预测值（真实值应为 1.5*5.0 + 0.6 = 8.1）

# 保存训练后的模型
torch.save(model.state_dict(), 'linear_regression_model.pth')

In [ ]:
# 打印训练后的模型参数
print("Model Parameters:")
for name, param in model.named_parameters():
    print(f"{name}: {param.detach().numpy()}")
    if name == 'linear.weight':  # 权值w
        w = list(param.detach().numpy())
    if name == 'linear.bias':    # 偏置b
        b = list(param.detach().numpy())

In [ ]:
"""【注意】运行前，请把下面代码中的“张三”改为自己的，123456改为自己的。"""
printLog('', )

# 4. 模型评估
##  损失函数
回归任务是对**连续值**的预测，希望模型能根据数据的特征输出一个连续值作为预测值。因此回归任务中常用的评估指标是**均方误差**。

令$\boldsymbol{y}\in \mathbb{R}^N$，$\hat{\boldsymbol{y}}\in \mathbb{R}^N$分别为$N$个样本的真实标签和预测标签，均方误差的定义为：

$$
\mathcal{L}(\boldsymbol{y},\hat{\boldsymbol{y}})=\frac{1}{2N}\|\boldsymbol{y}-\hat{\boldsymbol{y}}\|^2=\frac{1}{2N}\|\boldsymbol{X}\boldsymbol{w}+\boldsymbol{b}-\boldsymbol{y}\|^2, (2.8)
$$
其中$\boldsymbol{b}$为$N$维向量，所有元素取值都为$b$。

均方误差的代码实现如下:
> 注意：代码实现中没有除2。

In [ ]:
# 采用“HW2-1.机器学习概述-上.ipynb”中的均方误差函数mean_squared_error对训练好的模型在测试数据集上测试数据集X_test, y_test进行评估
# 或者采用torch的nn.MSELoss()函数对训练好的模型在测试数据集上测试数据集X_test, y_test进行评估
# 打印输出测试集上的平均误差
X_test_reshaped = X_test.reshape([-1, 1])
y_test_reshaped = y_test.reshape([-1, 1])
X_test_tensor = torch.tensor(X_test_reshaped, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_reshaped, dtype=torch.float32)

# 使用训练好的模型进行预测
model.eval()
with torch.no_grad():
    test_pred = model(X_test_tensor)
    # 使用PyTorch的MSELoss计算测试集误差
    test_loss = loss_fn(test_pred, y_test_tensor)

print(f"测试集上的均方误差 (PyTorch MSELoss): {test_loss.item():.6f}")

# 也可以使用自定义的均方误差函数
def mean_squared_error(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

# 使用numpy计算
test_pred_np = test_pred.numpy()
mse_numpy = mean_squared_error(y_test_reshaped, test_pred_np)
print(f"使用numpy计算的均方误差: {mse_numpy:.6f}")


In [ ]:
"""【注意】运行前，请把下面代码中的“张三”改为自己的，123456改为自己的。"""
printLog('', )